In [11]:
import pandas as pd

df = pd.read_csv("car_price_dataset.csv", sep=";")

print(df.columns.tolist())
print(df.head())

['Brand', 'Model', 'Year', 'Engine_Size', 'Fuel_Type', 'Transmission', 'Mileage', 'Doors', 'Owner_Count', 'Price']
        Brand   Model  Year  Engine_Size Fuel_Type    Transmission  Mileage  \
0         Kia     Rio  2020          4.2    Diesel          Manual   289944   
1   Chevrolet  Malibu  2012          2.0    Hybrid       Automatic     5356   
2    Mercedes     GLA  2020          4.2    Diesel       Automatic   231440   
3        Audi      Q5  2023          2.0  Electric          Manual   160971   
4  Volkswagen    Golf  2003          2.6    Hybrid  Semi-Automatic   286618   

   Doors  Owner_Count  Price  
0      3            5   8501  
1      2            3  12092  
2      4            2  11171  
3      2            1  11780  
4      3            3   2867  


In [12]:
X = df.drop(columns=["Price"])
y = df["Price"]

print(X.dtypes)

Brand               str
Model               str
Year              int64
Engine_Size     float64
Fuel_Type           str
Transmission        str
Mileage           int64
Doors             int64
Owner_Count       int64
dtype: object


In [13]:
from sklearn.preprocessing import OneHotEncoder

categorical_columns = ["Brand", "Model", "Fuel_Type", "Transmission"]
numeric_columns = ["Year", "Engine_Size", "Mileage", "Doors", "Owner_Count"]

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_categorical = encoder.fit_transform(X[categorical_columns])

print("Kategoriska features efter encoding:", X_categorical.shape[1])

Kategoriska features efter encoding: 47


In [15]:
import numpy as np

X_numeric = X[numeric_columns].to_numpy()

X_encoded = np.hstack([X_numeric, X_categorical])

print("Features totalt:", X_encoded.shape[1])

Features totalt: 52


In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_scaled = scaler.fit_transform(X_encoded)

print("Shape:", X_scaled.shape)

Shape: (10000, 52)


In [17]:
print("Medelvärde:", X_scaled.mean())
print("Standardavvikelse:", X_scaled.std())

Medelvärde: 2.120423494908699e-16
Standardavvikelse: 1.0000000000000038


In [18]:
print("Medelvärde:", X_scaled.mean())
print("Standardavvikelse:", X_scaled.std())

Medelvärde: 2.120423494908699e-16
Standardavvikelse: 1.0000000000000038


In [19]:
from sklearn.decomposition import PCA

pca = PCA(n_components=0.95)

X_pca = pca.fit_transform(X_scaled)

print("Features före PCA:", X_scaled.shape[1])
print("Komponenter efter PCA:", X_pca.shape[1])
print("Förklarad varians:", pca.explained_variance_ratio_.sum())

Features före PCA: 52
Komponenter efter PCA: 37
Förklarad varians: 0.9632823033258667


In [20]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42
)

X_pca_train, X_pca_test, y_pca_train, y_pca_test = train_test_split(
    X_pca,
    y,
    test_size=0.2,
    random_state=42
)

print("Original:", X_train.shape)
print("PCA:", X_pca_train.shape)

Original: (8000, 52)
PCA: (8000, 37)


In [21]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Dela först
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded,
    y,
    test_size=0.2,
    random_state=42
)

# Standardisering
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# PCA
pca = PCA(n_components=0.95)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

print("Utan PCA:", X_train_scaled.shape)
print("Med PCA:", X_train_pca.shape)
print("Förklarad varians:", pca.explained_variance_ratio_.sum())

Utan PCA: (8000, 52)
Med PCA: (8000, 37)
Förklarad varians: 0.9638651525133894


In [22]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

# Modell utan PCA
model_original = LinearRegression()
model_original.fit(X_train_scaled, y_train)

pred_original = model_original.predict(X_test_scaled)

rmse_original = np.sqrt(
    mean_squared_error(y_test, pred_original)
)

# Modell med PCA
model_pca = LinearRegression()
model_pca.fit(X_train_pca, y_train)

pred_pca = model_pca.predict(X_test_pca)

rmse_pca = np.sqrt(
    mean_squared_error(y_test, pred_pca)
)

print("RMSE utan PCA:", rmse_original)
print("RMSE med PCA:", rmse_pca)

RMSE utan PCA: 64.91473251816448
RMSE med PCA: 1562.7024455578903


Efter PCA reducerades antalet features från 52 till 37 samtidigt som 96,39 % av variansen behölls. Däremot försämrades modellens resultat kraftigt. RMSE ökade från 64,91 utan PCA till 1562,70 med PCA. Detta visar att PCA kan minska dimensionaliteten men samtidigt ta bort information som är viktig för prediktionen. PCA fokuserar på att bevara varians, inte på att maximera prediktionsförmågan